In [ ]:
import json
import time
import os
from pathlib import Path
from requests.exceptions import HTTPError
from web3 import Web3
from concurrent.futures import ThreadPoolExecutor, as_completed
from hexbytes import HexBytes
from web3.datastructures import AttributeDict
from ethscan_main import POOL_ADDRESS
from eth_abi import decode
from dotenv import load_dotenv

load_dotenv()  # Loads variables from .env into environment

INFURA_API_KEY = os.getenv("INFURA_API_KEY")
if not INFURA_API_KEY:
    raise Exception("INFURA_API_KEY not found in environment variables.")

# === CONFIGURATION ===
### URL
INFURA_URL = f"https://mainnet.infura.io/v3/{INFURA_API_KEY}"
CONTRACT_ADDRESS = POOL_ADDRESS = "0xCBCdF9626bC03E24f779434178A73a0B4bad62eD"
### FILES
ABI_FILE = "WETH_WBTC_pool.json"  # Load your contract ABI file
BLOCKS_FILE = "processed_blocks.json"
TRANSACTIONS_FILE = "transactions.json"
PROCESSED_TX_FILE = "processed_transaction_block.txt"
MINT_CALLS_FILE = "mint_calls.json"
### CONSTANTS
# Number of transactions to process before writing to disk
BATCH_SIZE = 200  
# Define how many blocks to fetch in one run
BATCH_BLOCK =200
# The contract was created on this block + 1 (12369821)
UNISWAP_CONTRACT_BLOCK_CREATION_NUMBER = 12369820

# === CONNECT TO ETHEREUM NODE ===
w3 = Web3(Web3.HTTPProvider(INFURA_URL))

assert w3.is_connected(), "Web3 provider connection failed"


# === UTILITY FUNCTIONS ===
# LOAD CONTRACT ABI
def load_contract_abi(file_path):
    """Load the contract ABI from a JSON file."""
    try:
        with open(file_path, "r") as f:
            return json.load(f)
    except FileNotFoundError:
        raise Exception(f"⚠️ ABI file '{file_path}' not found!")
    except json.JSONDecodeError:
        raise Exception(f"❌ Error parsing ABI JSON from '{file_path}'!")


contract_abi = load_contract_abi(ABI_FILE)
contract = w3.eth.contract(address=CONTRACT_ADDRESS, abi=contract_abi)

In [ ]:
class HexBytesEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, HexBytes):
            return obj.hex()
        if isinstance(obj, AttributeDict):
            return dict(obj)  # Convert AttributeDict to a regular dictionary
        return super().default(obj)


def load_blocks_data(file_path):
    """Load block data from a JSON file as a dictionary mapping block numbers to block data."""
    try:
        with open(file_path, "r") as f:
            data = json.load(f)
            # Convert keys to integers (JSON keys are strings)
            return {int(k): v for k, v in data.items()}
    except (FileNotFoundError, json.JSONDecodeError):
        return {}


def save_blocks_data(blocks_data, file_path):
    """Save the blocks_data dictionary to a JSON file."""
    # Convert keys back to strings for JSON serialization.
    data_to_save = {str(k): v for k, v in blocks_data.items()}
    with open(file_path, "w") as f:
        json.dump(data_to_save, f, indent=2)


def fetch_block(w3, block_number):
    """Fetch a single block with full transaction data using retries."""
    retries = 3
    while retries:
        try:
            block = w3.eth.get_block(block_number, full_transactions=True)
            print(f"Block {block_number} fetched with hash {block.hash.hex()}")
            # Process transactions: convert each transaction hash to a string.
            tx_hashes = []
            for tx in block.transactions:
                try:
                    # If tx is a dict with a 'hash' key
                    tx_hashes.append(tx["hash"].hex())
                except (TypeError, KeyError):
                    tx_hashes.append(tx.hex())
            return {
                "number": block.number,
                "hash": block.hash.hex(),
                "transactions": tx_hashes,
            }
        except HTTPError as e:
            if e.response.status_code == 429:
                print(f"Rate limit hit, retrying block {block_number} in 2 seconds...")
                time.sleep(2)
                retries -= 1
            else:
                print(f"Failed to fetch block {block_number}: {e}")
                raise
    print(f"Failed to fetch block {block_number} after retries")
    return None


def fetch_blocks(w3, blocks_file, batch_block):
    """
    Fetch new blocks in batches and ensure data integrity.

    The block data is stored in a single file (blocks_file) as a dictionary mapping:
        block_number -> block_data

    The function first loads existing block data. It then checks for missing blocks in the
    range from the first block (which should equal UNISWAP_CONTRACT_BLOCK_CREATION_NUMBER) to the
    highest block number already fetched. Any missing blocks are re-fetched.

    Then, new blocks are fetched starting from (max(existing_keys) + 1) up to batch_block count,
    or until the current chain's latest block.

    New blocks are written (streamed) every STREAM_BATCH blocks.
    """
    STREAM_BATCH = 200  # flush every 200 new blocks

    # Load existing block data as a dictionary {block_number: block_data}
    blocks_data = load_blocks_data(blocks_file)

    # Determine the starting block for re-fetching missing blocks:
    if blocks_data:
        first_block = min(blocks_data.keys())
        last_block = max(blocks_data.keys())
    else:
        # If no block has been fetched yet, start at the contract's creation block.
        first_block = UNISWAP_CONTRACT_BLOCK_CREATION_NUMBER
        last_block = first_block - 1

    # Integrity check: find missing blocks in the range [first_block, last_block]
    expected_range = set(range(first_block, last_block + 1))
    fetched_blocks = set(blocks_data.keys())
    missing_blocks = sorted(expected_range - fetched_blocks)
    if missing_blocks:
        print(
            f"Integrity check: Missing {len(missing_blocks)} blocks in range {first_block}-{last_block}: {missing_blocks}"
        )
        # Re-fetch missing blocks in parallel.
        from concurrent.futures import ThreadPoolExecutor, as_completed

        with ThreadPoolExecutor(max_workers=10) as executor:
            future_to_block = {
                executor.submit(fetch_block, w3, b): b for b in missing_blocks
            }
            for future in as_completed(future_to_block):
                b = future_to_block[future]
                result = future.result()
                if result:
                    blocks_data[b] = result
                    print(f"Recovered missing block {b}")
        # Save after re-fetching missing blocks.
        save_blocks_data(blocks_data, blocks_file)
        print("Integrity recovery completed: missing blocks have been re-fetched.")
    else:
        print("Integrity check passed: no missing blocks in the current range.")

    # Now, determine the starting block for new fetches:
    if blocks_data:
        new_start = max(blocks_data.keys()) + 1
    else:
        new_start = UNISWAP_CONTRACT_BLOCK_CREATION_NUMBER

    latest_chain_block = w3.eth.block_number
    if new_start > latest_chain_block:
        print("No new blocks to fetch.")
        return blocks_data

    new_end = min(new_start + batch_block - 1, latest_chain_block)
    new_blocks_range = list(range(new_start, new_end + 1))
    print(
        f"Fetching new blocks from {new_start} to {new_end} (expected {len(new_blocks_range)} blocks)"
    )

    new_blocks_fetched = 0
    new_batch = {}
    from concurrent.futures import ThreadPoolExecutor, as_completed

    with ThreadPoolExecutor(max_workers=10) as executor:
        future_to_block = {
            executor.submit(fetch_block, w3, b): b for b in new_blocks_range
        }
        for future in as_completed(future_to_block):
            b = future_to_block[future]
            result = future.result()
            if result:
                new_batch[b] = result
                new_blocks_fetched += 1
            if new_blocks_fetched % STREAM_BATCH == 0 and new_blocks_fetched:
                blocks_data.update(new_batch)
                save_blocks_data(blocks_data, blocks_file)
                print(f"Flushed {len(new_batch)} new blocks to {blocks_file}.")
                new_batch = {}
    # Merge any remaining new blocks.
    blocks_data.update(new_batch)
    save_blocks_data(blocks_data, blocks_file)
    print(f"Total new blocks fetched in this batch: {new_blocks_fetched}.")
    print("Fetching process completed.")
    return blocks_data

In [ ]:
# Example usage (in a Jupyter cell):
# Make sure UNISWAP_CONTRACT_BLOCK_CREATION_NUMBER is defined (e.g., 12369820)
updated_blocks = fetch_blocks(w3, BLOCKS_FILE, BATCH_BLOCK)

In [ ]:
print(len(updated_blocks))
a = list(updated_blocks.keys())
a.sort()
print(len(a))
print(a[0])
print(a[-1])
print(a[-1] - a[0])

In [ ]:
print(len(updated_blocks))
a = list(updated_blocks.keys())
a.sort()
print(len(a))
print(a[0])
print(a[-1])
print(a[-1] - a[0])

In [ ]:
print(len(updated_blocks))
a = list(updated_blocks.keys())
a.sort()
print(len(a))
print(a[0])
print(a[-1])
print(a[-1] - a[0])

In [ ]:
def load_transactions_data(file_path):
    """Load transactions data from a JSON file as a dictionary mapping tx hash to transaction data."""
    try:
        with open(file_path, "r") as f:
            data = json.load(f)
            # Convert keys to strings (they should already be strings)
            return {str(k): v for k, v in data.items()}
    except (FileNotFoundError, json.JSONDecodeError):
        return {}


def load_blocks_data(file_path):
    """Load block data from a JSON file as a dictionary mapping block numbers to block data."""
    try:
        with open(file_path, "r") as f:
            data = json.load(f)
            # Convert keys to integers (JSON keys are strings)
            return {int(k): v for k, v in data.items()}
    except (FileNotFoundError, json.JSONDecodeError):
        return {}


def save_transactions_data(tx_data, file_path):
    """Save the transactions data dictionary to a JSON file."""
    with open(file_path, "w") as f:
        json.dump(tx_data, f, indent=2, cls=HexBytesEncoder)


def fetch_transaction(w3, tx_hash):
    """Fetch a single transaction using retries."""
    retries = 3
    while retries:
        try:
            tx = w3.eth.get_transaction(tx_hash)
            # Convert AttributeDict to dict if needed.
            return dict(tx)
        except HTTPError as e:
            if e.response.status_code == 429:
                time.sleep(2)
                retries -= 1
            else:
                print(f"Failed to fetch transaction {tx_hash}: {e}")
                raise
    return None


def process_block_transactions(block, w3, tx_data, max_attempts=3):
    """
    For a given block (dict with a "transactions" list), fetch missing transactions.

    Retries up to max_attempts for any transactions that are still missing after a parallel fetch.
    Updates and returns the tx_data dictionary (mapping tx_hash -> tx data).
    """
    from concurrent.futures import ThreadPoolExecutor, as_completed

    missing_txs = [tx for tx in block.get("transactions", []) if tx not in tx_data]
    attempt = 0
    while missing_txs and attempt < max_attempts:
        print(
            f"Block {block['number']}: Attempt {attempt+1} to fetch {len(missing_txs)} missing transactions."
        )
        new_tx_for_block = {}
        with ThreadPoolExecutor(max_workers=10) as executor:
            future_to_tx = {
                executor.submit(fetch_transaction, w3, tx): tx for tx in missing_txs
            }
            for future in as_completed(future_to_tx):
                tx_hash = future_to_tx[future]
                result = future.result()
                if result:
                    new_tx_for_block[tx_hash] = result
        # Update our global transaction data with any newly fetched transactions.
        tx_data.update(new_tx_for_block)
        # Check again which transactions remain missing.
        missing_txs = [tx for tx in block.get("transactions", []) if tx not in tx_data]
        if missing_txs:
            print(
                f"Block {block['number']}: After attempt {attempt+1}, still missing {len(missing_txs)} transactions."
            )
        attempt += 1
    return tx_data


def process_transactions_per_block(w3, blocks_file, transactions_file):
    """
    Process transactions block-by-block.

    For each block stored in blocks_file (a JSON dictionary mapping block number -> block data),
    the function checks the transactions list, fetches any missing transactions (with retries),
    and flushes the updated transactions data to transactions_file immediately after each block.

    This ensures that even if the process is interrupted, all transactions from processed blocks are saved.
    """
    # Load block data (as dictionary mapping block_number -> block_data)
    blocks_data = load_blocks_data(blocks_file)
    if not blocks_data:
        print("No blocks data available.")
        return {}

    # Load already processed transactions.
    tx_data = load_transactions_data(transactions_file)

    # Process blocks in sorted order.
    sorted_blocks = sorted(blocks_data.keys())
    total_new = 0

    for block_number in sorted_blocks:
        block = blocks_data[block_number]
        block_tx_hashes = block.get("transactions", [])
        # Determine which transactions in this block haven't been processed.
        missing_txs = [tx for tx in block_tx_hashes if tx not in tx_data]
        if not missing_txs:
            print(f"Block {block_number} already fully processed.")
        else:
            print(
                f"Processing block {block_number} with {len(missing_txs)} missing transactions."
            )
            # Process missing transactions with integrity check after processing the block.
            tx_data = process_block_transactions(block, w3, tx_data)
            new_in_block = len([tx for tx in block_tx_hashes if tx in tx_data])
            print(f"Block {block_number}: Processed {new_in_block} transactions.")
            total_new += new_in_block

        # Flush the updated transaction data to disk after each block.
        save_transactions_data(tx_data, transactions_file)
        print(f"Flushed transactions from block {block_number} to {transactions_file}.")

    print(f"Total new transactions processed in this run: {total_new}.")
    return tx_data

In [ ]:
# Example usage (in a Jupyter Notebook cell):
# Make sure your blocks_file and transactions_file are set up as expected.
processed_tx = process_transactions_per_block(w3, BLOCKS_FILE, TRANSACTIONS_FILE)

In [ ]:
# Mint function signature and selector
MINT_FUNCTION_SIGNATURE = "mint(address,int24,int24,uint128,bytes)"
MINT_SELECTOR = Web3.keccak(text=MINT_FUNCTION_SIGNATURE)[:4].hex()


# ----- Helper Functions -----
def load_processed_tx_hashes(file_path):
    """Load processed transaction hashes from a text file (one per line)."""
    try:
        with open(file_path, "r") as f:
            return set(line.strip() for line in f if line.strip())
    except FileNotFoundError:
        return set()


def append_processed_tx_hashes(new_hashes, file_path):
    """Append a set of transaction hashes to a file, one per line."""
    with open(file_path, "a") as f:
        for tx in new_hashes:
            f.write(tx + "\n")


def append_mint_calls(mint_calls, file_path):
    """Append mint call results to the output file, one JSON object per line."""
    with open(file_path, "a") as f:
        for call in mint_calls:
            f.write(json.dumps(call) + "\n")


def load_transactions_data(file_path):
    """Load the transactions data from a JSON file (assumed to be a dict of tx_hash -> tx data)."""
    try:
        with open(file_path, "r") as f:
            return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        return {}


def process_single_transaction(tx, mint_selector):
    """
    Process a single transaction.
    If the transaction's input field starts with the mint selector, decode the parameters
    and return a dict with the mint call details; otherwise return None.
    """
    input_data = tx.get("input", "")
    if input_data and input_data.startswith(mint_selector):
        # Remove the "0x" and the selector (first 10 characters: "0x" + 8 hex digits)
        data_without_selector = input_data[10:]
        try:
            params = decode(
                ["address", "int24", "int24", "uint128", "bytes"],
                bytes.fromhex(data_without_selector),
            )
            return {
                "transaction_hash": tx.get("hash"),
                "blockNumber": tx.get("blockNumber"),
                "mint_args": {
                    "recipient": params[0],
                    "tickLower": params[1],
                    "tickUpper": params[2],
                    "amount": params[3],
                    "data": (
                        params[4].hex() if isinstance(params[4], bytes) else params[4]
                    ),
                },
            }
        except Exception as e:
            print(f"Error decoding mint call in tx {tx.get('hash')}: {e}")
            return None
    return None


def process_transactions_in_batches(
    w3, transactions_file, processed_tx_file, mint_calls_file, batch_size
):
    """
    Process transactions from transactions_file to find Mint calls.

    - Loads the transactions (as a dict mapping tx_hash -> tx data).
    - Loads already processed transaction hashes from processed_tx_file.
    - Iterates over transactions that haven't been processed.
    - In parallel, processes each transaction to see if it is a Mint call.
    - Every batch_size transactions processed, flush the results to mint_calls_file
      (one JSON object per line) and append the processed transaction hashes to processed_tx_file.
    """
    all_tx = load_transactions_data(transactions_file)
    processed = load_processed_tx_hashes(processed_tx_file)

    # Only process transactions that have not been processed yet.
    tx_list = [tx for tx in all_tx.values() if tx.get("hash") not in processed]
    print(f"Total transactions to process: {len(tx_list)}")

    processed_in_batch = set()
    mint_calls_batch = []
    total_processed = 0

    with ThreadPoolExecutor(max_workers=10) as executor:
        futures = {
            executor.submit(process_single_transaction, tx, MINT_SELECTOR): tx
            for tx in tx_list
        }
        for future in as_completed(futures):
            tx = futures[future]
            tx_hash = tx.get("hash")
            processed_in_batch.add(tx_hash)
            result = future.result()
            if result is not None:
                mint_calls_batch.append(result)
            total_processed += 1

            # If we've processed a batch, flush the results.
            if total_processed % batch_size == 0:
                if mint_calls_batch:
                    append_mint_calls(mint_calls_batch, mint_calls_file)
                    print(
                        f"Flushed {len(mint_calls_batch)} mint call entries to {mint_calls_file}."
                    )
                    mint_calls_batch = []
                if processed_in_batch:
                    append_processed_tx_hashes(processed_in_batch, processed_tx_file)
                    processed_in_batch = set()

    # Flush any remaining entries after processing all transactions.
    if mint_calls_batch:
        append_mint_calls(mint_calls_batch, mint_calls_file)
        print(
            f"Flushed remaining {len(mint_calls_batch)} mint call entries to {mint_calls_file}."
        )
    if processed_in_batch:
        append_processed_tx_hashes(processed_in_batch, processed_tx_file)
        print(
            f"Updated processed transaction file with remaining {len(processed_in_batch)} entries."
        )

    print("Transaction processing complete.")

In [ ]:
# ----- Run the Process in Your Jupyter Notebook Cell -----
process_transactions_in_batches(
    w3, TRANSACTIONS_FILE, PROCESSED_TX_FILE, MINT_CALLS_FILE, BATCH_SIZE
)